In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
import pypsa
import xlsxwriter
import tz_pypsa
import tz_pypsa.wrangle as wrangle
import pandas as pd
# import tz_solve
import plotly.express as px
import plotly.graph_objects as go
from tz_pypsa.model import Model
from tz_pypsa.utils import get_examples
import os
import glob

In [12]:
n = pypsa.Network()
n.import_from_netcdf("G:/Shared drives/Analysis/01 Projects/2024/Google 24 7 CFE/04. Japan/02. Data & Results/Outputs/1_Diagnosis/TP1/Run001/JPN_P1_JPN04/solved_networks/hourly_matching_CFE80_2030.nc")

INFO:pypsa.io:Imported network hourly_matching_CFE80_2030.nc has buses, carriers, generators, links, loads, storage_units


In [13]:
hourly_df = wrangle.transform_visualiser_hourly_output(n)

In [ ]:
hourly_df[hourly_df['Type']]=='Storage'

In [5]:
# ─── USER CONFIG ───────────────────────────────────────────────────────────────
# Path where all your .nc files live (e.g. configs["paths"]["..."])
INPUT_DIR = "G:/Shared drives/Analysis/01 Projects/2024/Google 24 7 CFE/04. Japan/02. Data & Results/Outputs/1_Diagnosis/TP1/Run001/JPN_P1_JPN04/solved_networks/"

# (Optional) Where you’d like to write the final combined CSV
OUTPUT_COMBINED_CSV = "/path/to/combined_output.csv"
# ────────────────────────────────────────────────────────────────────────────────


def list_nc_files(directory):
    """
    Return a sorted list of all .nc file‐paths under `directory`.
    """
    pattern = os.path.join(directory, "*.nc")
    files = sorted(glob.glob(pattern))
    return files

In [7]:
def postprocess_nc_list(nc_files):
    """
    Loop over every single file‐path in `nc_files`, run your PyPSA import + wrangle steps,
    collect the resulting hourly‐DataFrame for each file, then concatenate them all.

    Returns:
        combined_df (pd.DataFrame): one big DataFrame containing all per‐file results.
    """
    all_hourly_dfs = []

    for nc_path in nc_files:
        fname = os.path.basename(nc_path)
        print(f"▶ Processing {fname} …")

        # 1) Create a fresh PyPSA Network and import just this one .nc:
        n = pypsa.Network()
        n.import_from_netcdf(nc_path)

        # 2) Call your existing wrangle function to get a pandas DataFrame.
        #    (You said: wrangle.transform_visualiser_hourly_output(n) returns a DataFrame)
        hourly_df = wrangle.transform_visualiser_hourly_output(n)

        # 3) (Optional) Tag every row with the source filename, so you know
        #    which piece of data came from which .nc file later on:
        hourly_df["Scenario"] = fname

        # 4) Append to our list
        all_hourly_dfs.append(hourly_df)

        # 5) (Good practice) close the network’s internal file handle if needed
        #    (PyPSA’s Network() doesn’t strictly require a .close(), but if your
        #     pipeline opens files, you can always delete or garbage‐collect `n`.)
        del n

    # After looping through all files, concatenate into one big DataFrame:
    if not all_hourly_dfs:
        # If no files or something failed, return an empty DataFrame
        return pd.DataFrame()

    combined_df = pd.concat(all_hourly_dfs, axis=0, ignore_index=True)
    return combined_df

In [8]:
nc_files = list_nc_files(INPUT_DIR)

In [10]:
# 2) Run the postprocessing loop—this returns one large DataFrame
concat_df = postprocess_nc_list(nc_files)

# 3) Quick sanity check: Show how many rows & columns we ended up with
print(f"✔️  Combined DataFrame shape: {concat_df.shape}")

▶ Processing annual_matching_RES100_2030.nc …


INFO:pypsa.io:Imported network annual_matching_RES100_2030.nc has buses, carriers, generators, links, loads, storage_units


▶ Processing brownfield_2030.nc …


INFO:pypsa.io:Imported network brownfield_2030.nc has buses, carriers, generators, links, loads, storage_units


▶ Processing hourly_matching_CFE100_2030.nc …


...

▶ Processing hourly_matching_CFE90_2030.nc …


INFO:pypsa.io:Imported network hourly_matching_CFE90_2030.nc has buses, carriers, generators, links, loads, storage_units


▶ Processing hourly_matching_CFE95_2030.nc …


INFO:pypsa.io:Imported network hourly_matching_CFE95_2030.nc has buses, carriers, generators, links, loads, storage_units


✔️  Combined DataFrame shape: (35399160, 14)


In [12]:
concat_df.to_csv('Japan_TP1_Hourly_001.csv', index=False)

In [4]:
hourly_df = wrangle.transform_visualiser_hourly_output(n)

Please enter the market/country/region for this Pypsa run.
Examples: 'ASEAN, Japan, Taiwan, Indonesia, etc'
You entered: Japan
Please enter the run identifier.
Examples: '1, first-run, run-with-policy-constraint, etc'
You entered: Testing


In [5]:
yearly_df = wrangle.transform_visualiser_yearly_output(n)

Please enter the market/country/region for this Pypsa run.
Examples: 'ASEAN, Japan, Taiwan, Indonesia, etc'
You entered: Japan
Please enter the run identifier.
Examples: '1, first-run, run-with-policy-constraint, etc'
You entered: Testing


In [5]:
yearly_df.to_csv('japan_brownfield_yearly_008.csv', index=False)

In [6]:
hourly_df.to_csv('japan_brownfield_hourly_008.csv', index=False)